In [ ]:
import requests
url = 'https://data.lacity.org/resource/d5tf-ez2w.json?$limit=100'
r = requests.get(url, timeout=60)
r.raise_for_status()
data = r.json()
print('Fetched', len(data), 'records')
print('Keys:', list(data[0].keys()))

In [ ]:
FIELDS = ['dr_no','date_occ','time_occ','area_name','crm_cd_desc','vict_age','vict_sex','vict_descent','premis_desc','location','cross_street','lat','lon','mocodes']
rows = []
for rec in data:
    row = tuple(str(rec.get(f, '') or '') for f in FIELDS)
    rows.append(row)
print('Normalized', len(rows), 'rows')
print('Sample:', rows[0])

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F
schema = StructType([StructField(f, StringType(), True) for f in FIELDS])
df = spark.createDataFrame(rows, schema)
df.show(3, truncate=30)
print('Count:', df.count())

In [ ]:
df2 = df.withColumn('crash_year', F.year(F.to_date('date_occ')))
df2.write.format('delta').mode('overwrite').partitionBy('crash_year').saveAsTable('bronze_crashes')
print('Written!')
spark.sql('SELECT count(*) FROM bronze_crashes').show()

In [ ]:
print('BRONZE COMPLETE')